# Regime-Stratified Sensitivity Analysis — Level 5 (no surface kinetics)

Sensitivity of the **L5** model (defective oxide L3 + defective metal microstructure L4)
**conditioned on the rate-limiting regime**, run **isothermally** at several temperatures.
This is the no-surface-kinetics counterpart of `sensitivity_regime_L5L6.ipynb`.

## Regimes — oxide / metal / defect

Without a surface step there are only two series resistances, so the third competitor is the
**parallel** one: flux that bypasses the intact oxide through pinholes, cracks and
grain-boundary paths.

```
frac_oxide  = X_ox       * (1 - frac_defect)       series: degree of rate control,
frac_metal  = (1 - X_ox) * (1 - frac_defect)       X_ox = (P_up-P_int)/(P_up+P_int)
frac_defect =  J_defect / J_total                         parallel bypass
regime      = argmax of the three
```

The series split is each layer's degree of rate control, from the solved interface pressure
(`series_rate_control_L5`, shown for `P_downstream = 0`). The linear pressure drop
`(P_up - P_int)/P_up` is not used: it leans towards `oxide` because the metal runs on √P.

## Why isothermal

`temperature` enters `D`, `K_s`, `D_ox` and `K_ox` exponentially over 573-1273 K, so with it
varying it monopolises the variance of log10(flux). A dummy-parameter test on varying-T
clusters found `temperature` at delta ~0.50 and **every other parameter at or below the noise
floor (~0.09)** — a table of noise that reads like a ranking. Sampling harder does not help:
the floor is flat in n (0.085 at n=400, 0.092 at n=1500) because delta's bias for an
irrelevant input does not shrink with sample size. Pinning the temperature is what lets the
remaining parameters compete.

Every result table carries **`floor`** / **`floor_pawn`** columns (max delta / PAWN over 20
dummy inputs the model never saw) and the ratios **`delta_over_floor`** / **`pawn_over_floor`**.
Treat any parameter at or below 1.0x on both as unresolved.

## Targeting is not optional

Over the default ranges the regimes are badly unbalanced, so the presets create the
study. Yields are measured by a
probe at run time and printed in Phase 1.

> **Assumption to state in any write-up:** the `defect` preset allows a total defect area up
> to ~0.9 — a heavily degraded coating, a wider claim than the <=0.3 the default ranges imply.

## Sampling, regimes and reading the indices

- **Log-uniform sampling.** Every scale parameter (pre-exponentials, thicknesses, pressure,
  defect fractions, ratios, grain size) is drawn log-uniformly, so each decade of its range
  gets equal weight; activation energies and heats are drawn linearly. The cache check
  refuses scans saved under a different `SCAN_VERSION`.
- **Regime fractions are degrees of rate control**, $X_i = \partial\ln J/\partial\ln k_i$, so a
  label names the step whose speed-up would raise the flux most. They are unit-free and sum to 1.
- **The indices are regional.** A cluster is the set of draws from one preset that the model put
  in one regime. Every index is therefore conditional on that preset's ranges and on a
  selection made on the output, and describes sensitivity *within* the regime, not over the
  whole design space. A parameter the preset pins or narrows cannot score highly. Phase 4
  repeats the analysis on each preset's full scan, without the regime selection, to show what
  the selection itself does.
- **Two noise floors.** 20 dummy inputs the model never reads give a δ floor and a PAWN floor
  (`floor`, `floor_pawn`; ratios `delta_over_floor`, `pawn_over_floor`). δ's floor stays near
  0.07 however large the cluster; PAWN's falls roughly as 1/√n, which is why the clusters
  target ~5000 rows. **A ratio at or below 1.0 is not resolved.** Clearing both floors is the
  robust result.

Run with the **`mace_env`** kernel (SALib pinned to 1.5.0).

In [1]:
import os, sys, importlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
# This notebook must run with its OWN folder as the working directory: that is how
# calculations.config.model_config finds this folder's study_config.py, and how
# RESULTS_DIR below resolves. Fails loudly rather than silently using another study.
from calculations.config import assert_study_is_local
print(assert_study_is_local())

# Figures from this notebook go in one folder beside it.
from calculations.figures import results_dir, save_figure
FIGS = results_dir("sensitivity_regime_L5")

study: incoloy802_cr2o3  (incoloy802_cr2o3/study_config.py, via folder)


study: incoloy802_cr2o3  (incoloy802_cr2o3/study_config.py, via folder)


In [2]:
import calculations.sensitivity as sens
importlib.reload(sens)
from calculations.config.model_config import (
    SUGGESTED_RANGES_LEVEL5, REGIME_PRESETS_L5, DEFAULT_SWEEP_TEMPERATURES
)
from calculations.sensitivity import (
    REGIME_LABELS_L5, REGIME_SA_METRICS_L5, LOG_METRICS_L5, SCAN_OUTPUT_FIELDS_L5,
    SCAN_LABEL_FIELDS_L5, presets_without, level5_model_wrapper,
    run_targeted_regime_scans, load_regime_scans, plot_regime_exploration,
    givendata_sensitivity_by_regime, summarize_givendata, plot_givendata_results,
    regime_comparison_matrix, plot_regime_comparison_heatmap, plot_regime_geometry, SCAN_VERSION, scan_version
)

# temperature is PINNED, not sampled -> it leaves the SA input set entirely
PRESETS = presets_without(REGIME_PRESETS_L5, 'temperature')
PARAMS  = [p for p in SUGGESTED_RANGES_LEVEL5 if p != 'temperature']
print(f"{len(PARAMS)} sampled parameters (temperature pinned); regimes: {REGIME_LABELS_L5}")

18 sampled parameters (temperature pinned); regimes: ('oxide', 'metal', 'defect')


In [3]:
# === Run configuration ========================================================
SEED         = 42
TEMPERATURES = list(DEFAULT_SWEEP_TEMPERATURES)      # 773 / 1073 / 1273 K
METRICS      = REGIME_SA_METRICS_L5                  # ['flux', 'permeability', 'PRF']
TOP_K        = 8

# draws = target / measured yield, per temperature — yields shift with T, so a single
# dict would leave the worst-yielding cluster far smaller and reintroduce the very
# noise-floor imbalance the isothermal design is meant to remove.
# Draw counts are no longer precomputed from a cached yield table.
# Passing N_per_regime=None makes run_targeted_regime_scans probe each
# preset and size the run from the yield it actually observes, which
# stays correct if a range, a preset or the active study changes.
N_BY_T = {T: None for T in TEMPERATURES}

RESULTS_DIR     = "sa_results_L5"
FORCE_RECOMPUTE = False
os.makedirs(RESULTS_DIR, exist_ok=True)


## Phase 1 — Targeted per-regime scans, one set per temperature

Cached to `sa_results_L5/scans/T<K>/scan_<regime>.csv`. The cache is only reused when every
file holds exactly the draw count currently requested — existence alone would silently serve
a previous configuration.

In [4]:
def cache_is_valid(paths, n_expected):
    """Reusable only if every scan exists, and matches the count if one was asked for.

    With auto-sizing `n_expected` is None: the draw count is decided by the probe
    at run time, so existence is all that can be checked. Set FORCE_RECOMPUTE to
    re-probe and re-scan.
    """
    d = os.path.dirname(next(iter(paths.values())))
    if scan_version(d) != SCAN_VERSION:
        return False, (f"scans in {d} are version {scan_version(d)}, not {SCAN_VERSION}: "
                       f"drawn under an older sampling or labelling design")
    for r, p in paths.items():
        if not os.path.exists(p):
            return False, f"{os.path.basename(p)} missing"
        if n_expected is None:
            continue
        n_have = len(pd.read_csv(p, usecols=[0]))
        if n_have != n_expected[r]:
            return False, (f"{os.path.basename(p)} holds {n_have} draws but config "
                           f"asks for {n_expected[r]}")
    return True, "every scan matches the requested draw counts"


clusters_T, scans_T, yields_T = {}, {}, {}

for T in TEMPERATURES:
    scan_dir = os.path.join(RESULTS_DIR, "scans", f"T{int(T)}")
    os.makedirs(scan_dir, exist_ok=True)
    paths = {r: os.path.join(scan_dir, f"scan_{r}.csv") for r in REGIME_LABELS_L5}
    reusable, why = cache_is_valid(paths, N_BY_T[T])

    print(f"\n{'='*70}\nT = {T:.0f} K\n{'='*70}")
    if (not FORCE_RECOMPUTE) and reusable:
        print(f"  loading cached scans ({why})")
        cl, sc, _ = load_regime_scans(scan_dir, regimes=REGIME_LABELS_L5)
        yl = {r: len(cl[r]) / len(sc[r]) for r in cl}
    else:
        print("  recomputing —", "FORCE_RECOMPUTE is set" if FORCE_RECOMPUTE else why)
        cl, sc, _, yl = run_targeted_regime_scans(
            N_per_regime=N_BY_T[T], regimes=REGIME_LABELS_L5,
            presets=PRESETS, wrapper=level5_model_wrapper,
            output_fields=SCAN_OUTPUT_FIELDS_L5, label_fields=SCAN_LABEL_FIELDS_L5,
            fixed_params={'temperature': T},
            seed=SEED, save_dir=scan_dir, verbose=False)
    clusters_T[T], scans_T[T], yields_T[T] = cl, sc, yl
    print("  clusters:", {r: len(cl[r]) for r in REGIME_LABELS_L5})
    print("  yields:  ", {r: f"{yl[r]*100:.1f}%" for r in REGIME_LABELS_L5})

master = pd.concat(
    [cl.assign(temperature_K=T) for T, c in clusters_T.items() for cl in [pd.concat(c.values())]],
    ignore_index=True)
master.to_csv(os.path.join(RESULTS_DIR, "master_clusters.csv"), index=False)
print(f"\nmaster_clusters.csv: {len(master)} rows")


T = 773 K
  recomputing — scans in sa_results_L5/scans/T773 are version None, not 2: drawn under an older sampling or labelling design


  saved scan -> sa_results_L5/scans/T773/scan_oxide.csv


  saved scan -> sa_results_L5/scans/T773/scan_metal.csv


  saved scan -> sa_results_L5/scans/T773/scan_defect.csv
  clusters: {'oxide': 5821, 'metal': 5560, 'defect': 5899}
  yields:   {'oxide': '65.6%', 'metal': '48.7%', 'defect': '89.4%'}

T = 1073 K
  recomputing — scans in sa_results_L5/scans/T1073 are version None, not 2: drawn under an older sampling or labelling design


  saved scan -> sa_results_L5/scans/T1073/scan_oxide.csv


  saved scan -> sa_results_L5/scans/T1073/scan_metal.csv


  saved scan -> sa_results_L5/scans/T1073/scan_defect.csv
  clusters: {'oxide': 6069, 'metal': 6074, 'defect': 5643}
  yields:   {'oxide': '70.1%', 'metal': '64.6%', 'defect': '77.3%'}

T = 1273 K
  recomputing — scans in sa_results_L5/scans/T1273 are version None, not 2: drawn under an older sampling or labelling design


  saved scan -> sa_results_L5/scans/T1273/scan_oxide.csv


  saved scan -> sa_results_L5/scans/T1273/scan_metal.csv


  saved scan -> sa_results_L5/scans/T1273/scan_defect.csv
  clusters: {'oxide': 5865, 'metal': 6029, 'defect': 5802}
  yields:   {'oxide': '67.7%', 'metal': '69.6%', 'defect': '72.6%'}



master_clusters.csv: 52762 rows


In [5]:
# Median outputs by temperature and regime — the physical sanity check
print(master.groupby(['temperature_K', 'regime'])[METRICS].median().to_string())

                              flux  permeability          PRF
temperature_K regime                                         
773.0         defect  1.042959e-11  1.690364e-14     5.385212
              metal   3.983220e-10  1.351304e-14     1.003736
              oxide   3.126798e-11  4.145661e-16  4313.865914
1073.0        defect  5.510547e-10  1.594783e-12     4.337125
              metal   1.545658e-08  1.375235e-12     1.000825
              oxide   8.596258e-10  8.127464e-14  1395.073531
1273.0        defect  2.896830e-09  9.866680e-12     3.965701
              metal   7.898564e-08  9.435260e-12     1.000337
              oxide   3.054822e-09  7.080235e-13   958.594696


In [6]:
for T in TEMPERATURES:
    print(f"\n===== T = {T:.0f} K =====")
    _fig = plot_regime_exploration(clusters_T[T], output_metrics=tuple(METRICS),
                            scan_df=pd.concat(scans_T[T].values(), ignore_index=True),
                            cross_col='regime_hierarchy')
    save_figure(_fig, f"regime_exploration_{T:.0f}K", FIGS)


===== T = 773 K =====


findfont: Failed to find font weight bold for STIXSizeFiveSym, now using 400.


findfont: Failed to find font weight bold for cmmi10, now using 400.


findfont: Failed to find font weight bold for cmsy10, now using 400.


findfont: Failed to find font weight bold for cmr10, now using 400.


findfont: Failed to find font weight bold for cmtt10, now using 400.


... (6 outputs left out)


===== T = 1073 K =====



  regime (argmax)  ×  regime_hierarchy (solver):
regime_hierarchy  metal_limited  oxide_limited/defect_limited  oxide_limited/regime_intact_oxide  transition
regime                                                                                                      
defect                      746                          7210                                  1         614
metal                      8287                             0                                  0         360
oxide                         0                             0                               7072        1063



===== T = 1273 K =====



  regime (argmax)  ×  regime_hierarchy (solver):
regime_hierarchy  metal_limited  oxide_limited/defect_limited  oxide_limited/regime_intact_oxide  transition
regime                                                                                                      
defect                      986                          6423                                  0         732
metal                      9287                             0                                  0         369
oxide                         0                             0                               6509        1001


## Phase 2 — Route B: given-data PAWN + delta, per temperature

Zero extra model evaluations. Each `(regime, metric)` entry carries the dummy-parameter
noise `floor`; `delta_over_floor <= 1` means the parameter is **not resolved**.

In [7]:
gd_T, tidy_T = {}, {}
for T in TEMPERATURES:
    print(f"\n===== T = {T:.0f} K =====")
    gd_T[T] = givendata_sensitivity_by_regime(
        clusters_T[T], PARAMS, output_metrics=METRICS, methods=('pawn', 'delta'),
        seed=SEED, log_metrics=LOG_METRICS_L5, n_dummy=20, verbose=True)
    tidy_T[T] = summarize_givendata(gd_T[T], PARAMS).assign(temperature_K=T)

tidy = pd.concat(tidy_T.values(), ignore_index=True)
tidy.to_csv(os.path.join(RESULTS_DIR, "routeB_givendata.csv"), index=False)
print(f"\nrouteB_givendata.csv: {len(tidy)} rows")


===== T = 773 K =====


  [oxide/flux] n= 5821  floor δ=0.075 PAWN=0.041  top-δ: P_upstream (0.680)
  [oxide/permeability] n= 5821  floor δ=0.074 PAWN=0.038  top-δ: P_upstream (0.455)
  [oxide/PRF] n= 5821  floor δ=0.059 PAWN=0.049  top-δ: P_upstream (0.283)
  [metal/flux] n= 5560  floor δ=0.068 PAWN=0.044  top-δ: P_upstream (0.417)
  [metal/permeability] n= 5560  floor δ=0.071 PAWN=0.048  top-δ: D_ref (0.339)
  [metal/PRF] n= 5560  floor δ=0.113 PAWN=0.046  top-δ: oxide_thickness (0.119)
  [defect/flux] n= 5899  floor δ=0.068 PAWN=0.037  top-δ: P_upstream (0.449)
  [defect/permeability] n= 5899  floor δ=0.072 PAWN=0.037  top-δ: D_ref (0.331)
  [defect/PRF] n= 5899  floor δ=0.066 PAWN=0.040  top-δ: f_pinhole (0.355)

===== T = 1073 K =====


  [oxide/flux] n= 6069  floor δ=0.065 PAWN=0.037  top-δ: P_upstream (0.569)
  [oxide/permeability] n= 6069  floor δ=0.065 PAWN=0.038  top-δ: P_upstream (0.294)
  [oxide/PRF] n= 6069  floor δ=0.065 PAWN=0.037  top-δ: P_upstream (0.229)
  [metal/flux] n= 6074  floor δ=0.065 PAWN=0.039  top-δ: P_upstream (0.418)
  [metal/permeability] n= 6074  floor δ=0.068 PAWN=0.039  top-δ: D_ref (0.368)
  [metal/PRF] n= 6074  floor δ=0.122 PAWN=0.037  top-δ: P_upstream (0.150)
  [defect/flux] n= 5643  floor δ=0.078 PAWN=0.041  top-δ: P_upstream (0.447)
  [defect/permeability] n= 5643  floor δ=0.069 PAWN=0.042  top-δ: D_ref (0.336)
  [defect/PRF] n= 5643  floor δ=0.067 PAWN=0.044  top-δ: f_pinhole (0.279)

===== T = 1273 K =====


  [oxide/flux] n= 5865  floor δ=0.066 PAWN=0.039  top-δ: P_upstream (0.520)
  [oxide/permeability] n= 5865  floor δ=0.067 PAWN=0.038  top-δ: P_upstream (0.224)
  [oxide/PRF] n= 5865  floor δ=0.062 PAWN=0.038  top-δ: P_upstream (0.179)
  [metal/flux] n= 6029  floor δ=0.063 PAWN=0.035  top-δ: P_upstream (0.410)
  [metal/permeability] n= 6029  floor δ=0.063 PAWN=0.041  top-δ: D_ref (0.356)
  [metal/PRF] n= 6029  floor δ=0.132 PAWN=0.040  top-δ: P_upstream (0.144)
  [defect/flux] n= 5802  floor δ=0.075 PAWN=0.042  top-δ: P_upstream (0.449)
  [defect/permeability] n= 5802  floor δ=0.073 PAWN=0.038  top-δ: D_ref (0.336)
  [defect/PRF] n= 5802  floor δ=0.064 PAWN=0.042  top-δ: f_pinhole (0.253)

routeB_givendata.csv: 486 rows


In [8]:
# Drivers of log10(flux) -- EVERY parameter, no threshold. Two noise floors: the best
# delta and the best PAWN median scored by 20 dummy inputs the model never saw. A ratio
# at or below 1.0 is indistinguishable from noise. delta's floor barely shrinks with
# cluster size; PAWN's does, so at these cluster sizes PAWN resolves smaller effects.
for T in TEMPERATURES:
    for reg in REGIME_LABELS_L5:
        sub = (tidy[(tidy.metric == 'flux') & (tidy.temperature_K == T)
                    & (tidy.regime == reg)]
               .sort_values('delta_over_floor', ascending=False))
        if sub.empty:
            continue
        fd, fp = sub['floor'].iloc[0], sub['floor_pawn'].iloc[0]
        res = (sub.delta_over_floor > 1.0) | (sub.pawn_over_floor > 1.0)
        print(f"  T={T:.0f}K  {reg:8s} n={sub.n.iloc[0]}  floor: delta={fd:.3f} "
              f"PAWN={fp:.3f}  ({int(res.sum())}/{len(sub)} above a floor)")
        for _, r_ in sub.head(8).iterrows():
            ok = r_.delta_over_floor > 1.0 or r_.pawn_over_floor > 1.0
            print(f"        {r_.parameter:24s} delta={r_.delta:.3f} ({r_.delta_over_floor:.2f}x)"
                  f"  PAWN={r_.pawn_median:.3f} ({r_.pawn_over_floor:.2f}x)"
                  + ('' if ok else '   <- at/below noise'))
    print()

  T=773K  oxide    n=5821  floor: delta=0.075 PAWN=0.041  (7/18 above a floor)
        P_upstream               delta=0.680 (9.04x)  PAWN=0.571 (13.77x)
        K_s_ref                  delta=0.112 (1.49x)  PAWN=0.109 (2.63x)
        f_pinhole                delta=0.086 (1.14x)  PAWN=0.065 (1.56x)
        metal_thickness          delta=0.077 (1.03x)  PAWN=0.057 (1.37x)
        E_D_ox                   delta=0.071 (0.94x)  PAWN=0.026 (0.63x)   <- at/below noise
        H_sol_ox                 delta=0.065 (0.86x)  PAWN=0.038 (0.91x)   <- at/below noise
        H_s                      delta=0.064 (0.84x)  PAWN=0.049 (1.19x)
        E_D                      delta=0.063 (0.83x)  PAWN=0.042 (1.02x)
  T=773K  metal    n=5560  floor: delta=0.068 PAWN=0.044  (5/18 above a floor)
        P_upstream               delta=0.417 (6.16x)  PAWN=0.366 (8.30x)
        D_ref                    delta=0.202 (2.98x)  PAWN=0.228 (5.17x)
        K_s_ref                  delta=0.164 (2.42x)  PAWN=0.146 (3.31x

In [9]:
for reg in REGIME_LABELS_L5:
    _fig = plot_givendata_results(gd_T[TEMPERATURES[-1]], reg, 'flux', PARAMS, top_n=5)
    save_figure(_fig, f"givendata_{reg}_flux", FIGS)

## Phase 2 (cont.) — regime geometry (highest temperature)

Clusters projected on their two strongest resolved drivers. Dashed boxes are the 10-90
percentile band; overlap is the reminder that regime boundaries are curved, not rectangular.

In [10]:
T_show = TEMPERATURES[-1]
def regime_topk(reg, k=TOP_K, metric='flux', T=None):
    """Top-k parameters for one regime at one temperature, ranked by Route-B delta."""
    T = T_show if T is None else T
    sub = tidy[(tidy.temperature_K == T) & (tidy.regime == reg) & (tidy.metric == metric)]
    return sub.nlargest(k, 'delta')['parameter'].tolist()

for reg in REGIME_LABELS_L5:
    _fig = plot_regime_geometry(clusters_T[T_show], axes=regime_topk(reg)[:2],
                         log_x=True, log_y=True,
                         title=f"L5 regime geometry @ {T_show:.0f} K — top-2 delta drivers of '{reg}'")
    save_figure(_fig, f"regime_geometry_{reg}", FIGS)

## Phase 3 — Cross-regime comparison, per temperature

Columns are column-normalised so each regime spans a full 0-1 scale; the small corner number
is the raw index. Exported `compare_*.csv` files are always raw.

In [11]:
for T in TEMPERATURES:
    print(f"\n===== T = {T:.0f} K — delta, log10(flux) =====")
    _fig = plot_regime_comparison_heatmap(gd_T[T], 'flux', PARAMS, index='delta', top_union=3)
    save_figure(_fig, f"comparison_heatmap_{T:.0f}K", FIGS)
    for idx in ('delta', 'pawn_median'):
        for met in METRICS:
            mat = regime_comparison_matrix(gd_T[T], met, PARAMS, index=idx, top_union=3)
            if not mat.empty:
                mat.to_csv(os.path.join(RESULTS_DIR, f"compare_{idx}_{met}_T{int(T)}.csv"))


===== T = 773 K — delta, log10(flux) =====



===== T = 1073 K — delta, log10(flux) =====



===== T = 1273 K — delta, log10(flux) =====


In [12]:
# How does each parameter's margin over the noise floor move with temperature?
# Unfiltered: every parameter appears at every T, so a value near 1.0 reads as
# "at the noise floor" instead of vanishing into a NaN that looks like missing data.
piv = (tidy[tidy.metric == 'flux']
       .pivot_table(index=['regime', 'parameter'], columns='temperature_K',
                    values='delta_over_floor'))
piv.columns = [f"{c:.0f}K" for c in piv.columns]
piv['best'] = piv.max(axis=1)

print("delta_over_floor for log10(flux) — ALL parameters, no threshold")
print("  1.0 = exactly at the dummy-parameter noise floor; higher = resolved\n")
for reg in REGIME_LABELS_L5:
    fl = (tidy[(tidy.metric == 'flux') & (tidy.regime == reg)]
          .groupby('temperature_K')['floor'].first())
    print(f"[{reg}]  floor: " + "  ".join(f"{T:.0f}K={v:.3f}" for T, v in fl.items()))
    print(piv.loc[reg].sort_values('best', ascending=False).round(2).to_string())
    print()

piv.to_csv(os.path.join(RESULTS_DIR, "delta_over_floor_by_T.csv"))
print("saved delta_over_floor_by_T.csv")

delta_over_floor for log10(flux) — ALL parameters, no threshold
  1.0 = exactly at the dummy-parameter noise floor; higher = resolved

[oxide]  floor: 773K=0.075  1073K=0.065  1273K=0.066
                        773K  1073K  1273K  best
parameter                                       
P_upstream              9.04   8.71   7.83  9.04
K_s_ref                 1.49   1.62   1.57  1.62
f_pinhole               1.14   1.06   1.25  1.25
H_sol_ox                0.86   0.92   1.22  1.22
D_ref                   0.81   1.07   1.12  1.12
metal_thickness         1.03   0.96   0.84  1.03
grain_size              0.70   0.66   0.97  0.97
gb_thickness            0.70   0.75   0.97  0.97
E_D_ox                  0.94   0.72   0.79  0.94
D_ox_ref                0.62   0.89   0.77  0.89
H_s                     0.84   0.72   0.89  0.89
E_D                     0.83   0.88   0.89  0.89
f_gb_defect             0.81   0.78   0.88  0.88
gb_diffusivity_factor   0.68   0.86   0.82  0.86
K_ox_ref                0.71

## Phase 4 — Regional versus preset-wide indices

The clusters are selected on the output, so the indices above are regional (see the top of
the notebook). Here the same estimators run on each preset's **full** scan — every draw,
whatever regime it landed in, subsampled to the cluster size — which removes the selection
but keeps the preset. Where a parameter ranks very differently in the two, it is the regime
selection, not the input on its own, that is doing the work.

In [13]:
N_PW = 5000   # subsample the full scans to about the cluster size, so floors compare
pw_T = {}
for T in TEMPERATURES:
    full = {r: df.sample(min(len(df), N_PW), random_state=SEED).reset_index(drop=True)
            for r, df in scans_T[T].items()}
    gd_pw = givendata_sensitivity_by_regime(
        full, PARAMS, output_metrics=['flux'], methods=('pawn', 'delta'),
        seed=SEED, log_metrics=LOG_METRICS_L5, n_dummy=20, verbose=False)
    pw_T[T] = summarize_givendata(gd_pw, PARAMS).assign(temperature_K=T)
pw = pd.concat(pw_T.values(), ignore_index=True)
pw.to_csv(os.path.join(RESULTS_DIR, "routeB_presetwide.csv"), index=False)

key = ['temperature_K', 'regime', 'parameter']
cols = ['delta_over_floor', 'pawn_over_floor']
both = (tidy[tidy.metric == 'flux'].set_index(key)[cols]
        .join(pw.set_index(key)[cols], rsuffix='_preset'))
for T in TEMPERATURES:
    for reg in REGIME_LABELS_L5:
        print(f"\n[{reg} @ {T:.0f} K]  in-regime cluster vs whole '{reg}' preset scan "
              f"(log10 flux, x floor)")
        print(both.loc[(T, reg)].sort_values('delta_over_floor', ascending=False)
              .head(6).round(2).to_string())


[oxide @ 773 K]  in-regime cluster vs whole 'oxide' preset scan (log10 flux, x floor)
                 delta_over_floor  pawn_over_floor  delta_over_floor_preset  pawn_over_floor_preset
parameter                                                                                          
P_upstream                   9.04            13.77                     9.52                   13.43
K_s_ref                      1.49             2.63                     1.10                    1.59
f_pinhole                    1.14             1.56                     1.10                    1.13
metal_thickness              1.03             1.37                     0.73                    0.97
E_D_ox                       0.94             0.63                     0.88                    0.69
H_sol_ox                     0.86             0.91                     0.76                    1.26

[metal @ 773 K]  in-regime cluster vs whole 'metal' preset scan (log10 flux, x floor)
                 delta_ove

In [14]:
# Manifest of everything saved
import glob
print("Saved CSV files under", os.path.abspath(RESULTS_DIR), ":")
for f in sorted(glob.glob(os.path.join(RESULTS_DIR, "**", "*.csv"), recursive=True)):
    print("  ", os.path.relpath(f, RESULTS_DIR), f"({os.path.getsize(f)//1024} KB)")

Saved CSV files under /Users/rwest/CoMoChEng Dropbox/Richard West/Northeastern/Research/Research-MHI-2026-hydrogen/Continuum_H_permeation/incoloy802_cr2o3/sa_results_L5 :
   compare_delta_PRF_T1073.csv (0 KB)
   compare_delta_PRF_T1273.csv (0 KB)
   compare_delta_PRF_T773.csv (0 KB)
   compare_delta_flux_T1073.csv (0 KB)
   compare_delta_flux_T1273.csv (0 KB)
   compare_delta_flux_T773.csv (0 KB)
   compare_delta_permeability_T1073.csv (0 KB)
   compare_delta_permeability_T1273.csv (0 KB)
   compare_delta_permeability_T773.csv (0 KB)
   compare_pawn_median_PRF_T1073.csv (0 KB)
   compare_pawn_median_PRF_T1273.csv (0 KB)
   compare_pawn_median_PRF_T773.csv (0 KB)
   compare_pawn_median_flux_T1073.csv (0 KB)
   compare_pawn_median_flux_T1273.csv (0 KB)
   compare_pawn_median_flux_T773.csv (0 KB)
   compare_pawn_median_permeability_T1073.csv (0 KB)
   compare_pawn_median_permeability_T1273.csv (0 KB)
   compare_pawn_median_permeability_T773.csv (0 KB)
   delta_over_floor_by_T.csv (4 KB)
 